# 气象数据探索性分析 (EDA)

本笔记本旨在对 `st-missing-fill` 项目中的气象数据进行全面的统计与可视化分析。

**分析维度：**
1. **站点元数据分析**：站点的空间分布、海拔特征及类型统计。
2. **变量元数据分析**：气象参数的覆盖情况。
3. **时间序列与质量分析**：以温度数据为例，探索缺失值模式、相关性及时间变化趋势。

---

In [ ]:
# 1. 环境设置与依赖加载
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster

# 设置绘图风格 - 优雅美观
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # 解决Mac中文显示问题
plt.rcParams['axes.unicode_minus'] = False

# 设置项目根目录
dir_path = '/Users/lxx/Documents/codes/st-missing-fill'
os.chdir(dir_path)
if dir_path not in sys.path:
    sys.path.append(dir_path)

print(f"工作目录已切换至: {os.getcwd()}")

## 2. 数据加载

加载站点信息、参数说明以及处理后的温度数据作为分析样本。

In [ ]:
# 加载元数据
stations = pd.read_parquet('data/meteo-swiss-description/stations.parquet')
params = pd.read_parquet('data/meteo-swiss-description/parameters.parquet')

# 加载温度数据样本 (2020-2025 处理后的数据)
temp_df = pd.read_parquet('data/processed-data/temperature.parquet')

print(f"站点数量: {len(stations)}")
print(f"参数数量: {len(params)}")
print(f"温度数据维度: {temp_df.shape}")
display(stations.head(3))
display(params.head(3))

## 3. 站点元数据分析

探索站点的地理特征（海拔、经纬度）和类型分布。

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 3.1 站点海拔分布
sns.histplot(data=stations, x='station_height', kde=True, color='skyblue', ax=ax[0])
ax[0].set_title('站点海拔分布直方图', fontsize=14, fontweight='bold')
ax[0].set_xlabel('海拔 (米)')
ax[0].set_ylabel('站点数量')

# 3.2 站点类型统计
type_counts = stations['station_type'].value_counts()
sns.barplot(x=type_counts.index, y=type_counts.values, palette='viridis', ax=ax[1])
ax[1].set_title('站点类型统计', fontsize=14, fontweight='bold')
ax[1].set_xlabel('站点类型')
ax[1].set_ylabel('数量')
for i, v in enumerate(type_counts.values):
    ax[1].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3.3 站点空间分布交互地图
m = folium.Map(
    location=[stations['latitude'].mean(), stations['longitude'].mean()],
    zoom_start=8,
    tiles='CartoDB positron'  # 使用简洁的底图
)

marker_cluster = MarkerCluster().add_to(m)

for idx, row in stations.iterrows():
    # 根据海拔设置颜色
    color = '#2ecc71' if row['station_height'] < 1000 else ('#f1c40f' if row['station_height'] < 2000 else '#e74c3c')
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        popup=f"<b>{row['station_name']}</b><br>海拔: {row['station_height']}m<br>类型: {row.get('station_type', 'N/A')}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
    ).add_to(marker_cluster)

# 添加图例说明 (HTML)
legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; left: 50px; width: 120px; height: 90px; 
     border:2px solid grey; z-index:9999; font-size:12px;
     background-color:white; opacity: 0.8; padding: 10px;">
     <b>海拔图例</b><br>
     <i style="background:#2ecc71;width:10px;height:10px;display:inline-block;border-radius:50%;"></i> < 1000m<br>
     <i style="background:#f1c40f;width:10px;height:10px;display:inline-block;border-radius:50%;"></i> 1000-2000m<br>
     <i style="background:#e74c3c;width:10px;height:10px;display:inline-block;border-radius:50%;"></i> > 2000m
      </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

m

## 4. 时间序列与数据质量分析 (以温度数据为例)

分析数据的完整性，并探索时间变化的规律。

In [ ]:
# 4.1 缺失值热力图 (选取前50个站点和最近1个月的数据进行展示)
subset_df = temp_df.iloc[-4320:, :50]  # 最近30天 (10min频率 -> 6*24*30 = 4320)

plt.figure(figsize=(20, 8))
sns.heatmap(subset_df.isnull().T, cmap='magma', cbar=False, xticklabels=False)
plt.title('温度数据缺失值分布热力图 (最近30天, 前50个站点)', fontsize=16, fontweight='bold')
plt.xlabel('时间 (索引)')
plt.ylabel('站点')
plt.show()

In [ ]:
# 4.2 整体缺失率统计
missing_rates = temp_df.isnull().mean() * 100

plt.figure(figsize=(12, 5))
sns.histplot(missing_rates, bins=30, color='salmon', kde=True)
plt.title('各站点温度数据缺失率分布', fontsize=14, fontweight='bold')
plt.xlabel('缺失率 (%)')
plt.ylabel('站点数量')
plt.axvline(x=missing_rates.mean(), color='red', linestyle='--', label=f'平均缺失率: {missing_rates.mean():.2f}%')
plt.legend()
plt.show()

In [ ]:
# 4.3 站点间温度相关性分析 (选取完整度最高的前10个站点)
top_stations = missing_rates.sort_values().index[:10]
corr_matrix = temp_df[top_stations].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', 
            vmin=0.8, vmax=1.0, square=True, linewidths=.5)
plt.title('Top 10 完整站点温度相关性矩阵', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# 4.4 季节性与日变化分析 (选取第一个站点)
sample_station = temp_df.columns[0]
sample_series = temp_df[sample_station].dropna()

# 添加时间特征
df_plot = pd.DataFrame({'Temperature': sample_series})
df_plot['Hour'] = sample_series.index.hour
df_plot['Month'] = sample_series.index.month

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 日变化箱线图
sns.boxplot(data=df_plot, x='Hour', y='Temperature', palette='coolwarm', ax=ax[0])
ax[0].set_title(f'站点 {sample_station} 温度日变化规律', fontsize=12, fontweight='bold')
ax[0].set_xlabel('小时 (0-23)')
ax[0].set_ylabel('温度 (°C)')

# 月变化箱线图
sns.boxplot(data=df_plot, x='Month', y='Temperature', palette='Spectral', ax=ax[1])
ax[1].set_title(f'站点 {sample_station} 温度季节性变化规律', fontsize=12, fontweight='bold')
ax[1].set_xlabel('月份')
ax[1].set_ylabel('温度 (°C)')

plt.tight_layout()
plt.show()